### 1.1 Abstract Product — `MenuItem`

In [1]:
from abc import ABC, abstractmethod

class MenuItem(ABC):
    """Base abstract class untuk semua item menu."""

    def __init__(self, name: str, price: int):
        self._name  = name
        self._price = price

    @abstractmethod
    def get_category(self) -> str:
        """Wajib diimplementasi oleh subclass (abstract method)."""
        pass

    def get_name(self)  -> str: return self._name
    def get_price(self) -> int: return self._price

    def __str__(self) -> str:
        return f'[{self.get_category()}] {self.get_name()} - Rp{self.get_price():,}'

print('OK: Class MenuItem (abstract) berhasil didefinisikan')


OK: Class MenuItem (abstract) berhasil didefinisikan


### 1.2 Concrete Products — `FoodItem` & `DrinkItem`

In [2]:
class FoodItem(MenuItem):
    """Produk konkret: item makanan."""
    def get_category(self) -> str:
        return 'Makanan'

class DrinkItem(MenuItem):
    """Produk konkret: item minuman."""
    def get_category(self) -> str:
        return 'Minuman'

print('OK: FoodItem dan DrinkItem berhasil didefinisikan')


OK: FoodItem dan DrinkItem berhasil didefinisikan


### 1.3 Factory (Creator) — `MenuFactory`

In [3]:
class MenuFactory:
    """
    Factory Method: menyembunyikan logika instantiasi dari client.
    Client cukup memanggil MenuFactory.create() tanpa perlu
    tahu FoodItem / DrinkItem secara langsung.
    """
    _registry: dict = {
        'makanan': FoodItem,
        'minuman': DrinkItem,
    }

    @classmethod
    def create(cls, category: str, name: str, price: int) -> MenuItem:
        category   = category.lower()
        item_class = cls._registry.get(category)
        if item_class is None:
            valid = ', '.join(cls._registry.keys())
            raise ValueError(f"Kategori '{category}' tidak dikenal. Pilihan valid: {valid}")
        return item_class(name, price)

    @classmethod
    def register(cls, category: str, item_class: type) -> None:
        """Extensibility: daftarkan jenis menu baru tanpa ubah kode factory."""
        cls._registry[category.lower()] = item_class

print('OK: MenuFactory berhasil didefinisikan')


OK: MenuFactory berhasil didefinisikan


### 1.4 Demo — Membuat item menu via Factory

In [4]:
nasi_goreng = MenuFactory.create('makanan', 'Nasi Goreng Spesial', 25_000)
kopi_hitam  = MenuFactory.create('minuman', 'Kopi Hitam',          15_000)
mie_ayam    = MenuFactory.create('makanan', 'Mie Ayam Bakso',      20_000)
es_teh      = MenuFactory.create('minuman', 'Es Teh Manis',         8_000)

for item in [nasi_goreng, kopi_hitam, mie_ayam, es_teh]:
    print(f'  OK  {item}')


  OK  [Makanan] Nasi Goreng Spesial - Rp25,000
  OK  [Minuman] Kopi Hitam - Rp15,000
  OK  [Makanan] Mie Ayam Bakso - Rp20,000
  OK  [Minuman] Es Teh Manis - Rp8,000


### 1.5 Extensibility — Daftarkan kategori baru tanpa ubah Factory

In [5]:
class DessertItem(MenuItem):
    def get_category(self) -> str:
        return 'Dessert'

MenuFactory.register('dessert', DessertItem)
puding = MenuFactory.create('dessert', 'Puding Coklat', 12_000)
print(f'  OK  {puding}  <- ditambahkan tanpa mengubah kode Factory')


  OK  [Dessert] Puding Coklat - Rp12,000  <- ditambahkan tanpa mengubah kode Factory


### 1.6 Guard — Kategori tidak dikenal

In [6]:
try:
    MenuFactory.create('camilan', 'Kerupuk', 2_000)
except ValueError as e:
    print(f'  ERROR: {e}')


  ERROR: Kategori 'camilan' tidak dikenal. Pilihan valid: makanan, minuman, dessert


---
<a id='2'></a>
## Pattern 2 — Decorator (Structural)

**Masalah:** Pelanggan ingin menambahkan topping ke menu secara dinamis.
Tanpa Decorator, tiap kombinasi butuh subclass tersendiri — *class explosion*
(2^4 = 16 subclass untuk 4 topping).

**Solusi:** Setiap topping *membungkus (wrap)* item sebelumnya,
mengakumulasi nama dan harga secara otomatis.

```
item = DrinkItem('Kopi', 15000)
item = MilkDecorator(item)       # +3.000 -> 18.000
item = SugarDecorator(item)      # +1.000 -> 19.000
item = ExtraShot(item)           # +7.000 -> 26.000
```


### 2.1 Abstract Decorator — `MenuDecorator`

In [7]:
class MenuDecorator(MenuItem, ABC):
    """
    Base Decorator: mewarisi MenuItem (agar bisa menggantikan item asli)
    sekaligus menyimpan referensi ke item yang dibungkusnya (_wrapped).
    Dua relasi ini adalah inti dari Decorator Pattern.
    """
    def __init__(self, wrapped_item: MenuItem):
        self._wrapped = wrapped_item  # tidak panggil super().__init__()

    def get_name(self)     -> str: return self._wrapped.get_name()
    def get_price(self)    -> int: return self._wrapped.get_price()
    def get_category(self) -> str: return self._wrapped.get_category()

    @abstractmethod
    def get_topping_name(self)  -> str: pass

    @abstractmethod
    def get_topping_price(self) -> int: pass

print('OK: MenuDecorator (abstract) berhasil didefinisikan')


OK: MenuDecorator (abstract) berhasil didefinisikan


### 2.2 Concrete Decorators

In [8]:
class MilkDecorator(MenuDecorator):
    """Topping susu — +Rp3.000"""
    TOPPING_NAME = 'Susu'; TOPPING_PRICE = 3_000
    def get_topping_name(self)  -> str: return self.TOPPING_NAME
    def get_topping_price(self) -> int: return self.TOPPING_PRICE
    def get_name(self)  -> str: return f'{self._wrapped.get_name()} + {self.TOPPING_NAME}'
    def get_price(self) -> int: return self._wrapped.get_price() + self.TOPPING_PRICE

class SugarDecorator(MenuDecorator):
    """Topping gula — +Rp1.000"""
    TOPPING_NAME = 'Gula'; TOPPING_PRICE = 1_000
    def get_topping_name(self)  -> str: return self.TOPPING_NAME
    def get_topping_price(self) -> int: return self.TOPPING_PRICE
    def get_name(self)  -> str: return f'{self._wrapped.get_name()} + {self.TOPPING_NAME}'
    def get_price(self) -> int: return self._wrapped.get_price() + self.TOPPING_PRICE

class CheeseDecorator(MenuDecorator):
    """Topping keju — +Rp5.000"""
    TOPPING_NAME = 'Keju'; TOPPING_PRICE = 5_000
    def get_topping_name(self)  -> str: return self.TOPPING_NAME
    def get_topping_price(self) -> int: return self.TOPPING_PRICE
    def get_name(self)  -> str: return f'{self._wrapped.get_name()} + {self.TOPPING_NAME}'
    def get_price(self) -> int: return self._wrapped.get_price() + self.TOPPING_PRICE

class ExtraShot(MenuDecorator):
    """Topping extra espresso shot — +Rp7.000"""
    TOPPING_NAME = 'Extra Shot'; TOPPING_PRICE = 7_000
    def get_topping_name(self)  -> str: return self.TOPPING_NAME
    def get_topping_price(self) -> int: return self.TOPPING_PRICE
    def get_name(self)  -> str: return f'{self._wrapped.get_name()} + {self.TOPPING_NAME}'
    def get_price(self) -> int: return self._wrapped.get_price() + self.TOPPING_PRICE

print('OK: MilkDecorator, SugarDecorator, CheeseDecorator, ExtraShot berhasil didefinisikan')


OK: MilkDecorator, SugarDecorator, CheeseDecorator, ExtraShot berhasil didefinisikan


### 2.3 Demo — Menumpuk Decorator lapis demi lapis

In [9]:
kopi = MenuFactory.create('minuman', 'Kopi Hitam', 15_000)
print(f'  Base   : {kopi.get_name():<45} Rp{kopi.get_price():,}')

kopi_susu = MilkDecorator(kopi)
print(f'  +Susu  : {kopi_susu.get_name():<45} Rp{kopi_susu.get_price():,}')

kopi_susu_gula = SugarDecorator(kopi_susu)
print(f'  +Gula  : {kopi_susu_gula.get_name():<45} Rp{kopi_susu_gula.get_price():,}')

kopi_full = ExtraShot(kopi_susu_gula)
print(f'  +Extra : {kopi_full.get_name():<45} Rp{kopi_full.get_price():,}')

print('\n  Visualisasi tumpukan Decorator:')
print(f'  kopi                           Rp15.000')
print(f'    MilkDecorator  (+3.000)  ->  Rp{kopi_susu.get_price():,}')
print(f'      SugarDecorator (+1.000) ->  Rp{kopi_susu_gula.get_price():,}')
print(f'        ExtraShot    (+7.000) ->  Rp{kopi_full.get_price():,}')


  Base   : Kopi Hitam                                    Rp15,000
  +Susu  : Kopi Hitam + Susu                             Rp18,000
  +Gula  : Kopi Hitam + Susu + Gula                      Rp19,000
  +Extra : Kopi Hitam + Susu + Gula + Extra Shot         Rp26,000

  Visualisasi tumpukan Decorator:
  kopi                           Rp15.000
    MilkDecorator  (+3.000)  ->  Rp18,000
      SugarDecorator (+1.000) ->  Rp19,000
        ExtraShot    (+7.000) ->  Rp26,000


### 2.4 Demo — Topping pada makanan

In [10]:
nasi = MenuFactory.create('makanan', 'Nasi Goreng Spesial', 25_000)
nasi_keju = CheeseDecorator(nasi)

print(f'  Base  : {nasi.get_name():<45} Rp{nasi.get_price():,}')
print(f'  +Keju : {nasi_keju.get_name():<45} Rp{nasi_keju.get_price():,}')


  Base  : Nasi Goreng Spesial                           Rp25,000
  +Keju : Nasi Goreng Spesial + Keju                    Rp30,000


---
<a id='3'></a>
## Pattern 3 — Observer (Behavioral)

**Masalah:** Saat order dikonfirmasi, banyak pihak (dapur, kasir, display antrian)
perlu diberitahu. Jika `OrderSubject` memanggil mereka langsung,
ia *tightly coupled* ke setiap penerima.

**Solusi:** `OrderSubject` adalah *Subject*. Dapur/kasir/display adalah *Observer*.
Subject hanya memanggil `_notify()`. Observer mendaftarkan diri sendiri.
Subject tidak tahu siapa yang mendengarkan *(loose coupling)*.

```
order.attach(KitchenObserver())
order.attach(CashierObserver())
order.confirm()  ->  _notify()  ->  semua observer bereaksi otomatis
```


### 3.1 Data class `OrderItem` & Abstract `Observer`

In [11]:
from __future__ import annotations
from dataclasses import dataclass
from datetime import datetime
from typing import List

@dataclass
class OrderItem:
    """Satu baris dalam order: item menu + jumlah."""
    item: MenuItem
    quantity: int = 1

    def subtotal(self) -> int:
        return self.item.get_price() * self.quantity

    def __str__(self) -> str:
        return (f'  {self.quantity}x {self.item.get_name():<35} '
                f'Rp{self.subtotal():>8,}')

class Observer(ABC):
    """
    Interface abstract untuk semua observer.
    Setiap observer WAJIB mengimplementasikan update().
    """
    @abstractmethod
    def update(self, order: OrderSubject) -> None:
        pass

print('OK: OrderItem dan Observer (abstract) berhasil didefinisikan')


OK: OrderItem dan Observer (abstract) berhasil didefinisikan


### 3.2 Subject — `OrderSubject`

In [12]:
class OrderSubject:
    """
    Subject: merepresentasikan satu order pelanggan.
    Menyimpan daftar observer dan memnotifikasi semua
    saat status order berubah.
    """
    STATUS_PENDING   = 'MENUNGGU'
    STATUS_CONFIRMED = 'DIKONFIRMASI'
    STATUS_COOKING   = 'DIMASAK'
    STATUS_READY     = 'SIAP'
    STATUS_DONE      = 'SELESAI'
    _order_counter   = 0

    def __init__(self, customer_name: str):
        OrderSubject._order_counter += 1
        self.order_id      = OrderSubject._order_counter
        self.customer_name = customer_name
        self.items: List[OrderItem] = []
        self.status        = self.STATUS_PENDING
        self.created_at    = datetime.now()
        self._observers: List[Observer] = []

    def attach(self, observer: Observer) -> None:
        if observer not in self._observers:
            self._observers.append(observer)

    def detach(self, observer: Observer) -> None:
        self._observers.remove(observer)

    def _notify(self) -> None:
        for obs in self._observers:
            obs.update(self)

    def add_item(self, item: MenuItem, quantity: int = 1) -> None:
        self.items.append(OrderItem(item, quantity))

    def total(self) -> int:
        return sum(oi.subtotal() for oi in self.items)

    def confirm(self)       -> None: self.status = self.STATUS_CONFIRMED; self._notify()
    def start_cooking(self) -> None: self.status = self.STATUS_COOKING;   self._notify()
    def mark_ready(self)    -> None: self.status = self.STATUS_READY;     self._notify()
    def complete(self)      -> None: self.status = self.STATUS_DONE;      self._notify()

    def __str__(self) -> str:
        lines = [
            f'  Order #{self.order_id} - {self.customer_name}',
            f'  Waktu  : {self.created_at.strftime("%H:%M:%S")}',
            f'  Status : {self.status}',
            '  ' + '-' * 48,
        ]
        for oi in self.items:
            lines.append(str(oi))
        lines.append('  ' + '-' * 48)
        lines.append(f"  {'TOTAL':<40} Rp{self.total():>8,}")
        return '\n'.join(lines)

print('OK: OrderSubject berhasil didefinisikan')


OK: OrderSubject berhasil didefinisikan


### 3.3 Concrete Observers

In [13]:
class KitchenObserver(Observer):
    """Dapur: bereaksi saat DIKONFIRMASI atau DIMASAK."""
    def update(self, order: OrderSubject) -> None:
        if order.status in (OrderSubject.STATUS_CONFIRMED, OrderSubject.STATUS_COOKING):
            print(f'    [DAPUR]   Order #{order.order_id} ({order.customer_name}) masuk - mulai memasak!')
            for oi in order.items:
                print(f'         -> {oi.quantity}x {oi.item.get_name()}')

class CashierObserver(Observer):
    """Kasir: bereaksi saat DIKONFIRMASI dan SELESAI."""
    def update(self, order: OrderSubject) -> None:
        if order.status == OrderSubject.STATUS_CONFIRMED:
            print(f'    [KASIR]   Order #{order.order_id} dikonfirmasi - total Rp{order.total():,} siap ditagih.')
        elif order.status == OrderSubject.STATUS_DONE:
            print(f'    [KASIR]   Order #{order.order_id} selesai - transaksi ditutup.')

class DisplayObserver(Observer):
    """Display antrian: bereaksi pada setiap perubahan status."""
    _STATUS_ICON = {
        OrderSubject.STATUS_CONFIRMED: '[KUNING]',
        OrderSubject.STATUS_COOKING:   '[BIRU]',
        OrderSubject.STATUS_READY:     '[HIJAU]',
        OrderSubject.STATUS_DONE:      '[PUTIH]',
    }
    def update(self, order: OrderSubject) -> None:
        icon = self._STATUS_ICON.get(order.status, '[?]')
        print(f'    [DISPLAY] Nomor antrian #{order.order_id} - {icon} {order.status}')

print('OK: KitchenObserver, CashierObserver, DisplayObserver berhasil didefinisikan')


OK: KitchenObserver, CashierObserver, DisplayObserver berhasil didefinisikan


### 3.4 Demo — Order dengan notifikasi otomatis

In [14]:
order_1 = OrderSubject('Budi Santoso')
order_1.attach(KitchenObserver())
order_1.attach(CashierObserver())
order_1.attach(DisplayObserver())

item_a = SugarDecorator(MilkDecorator(MenuFactory.create('minuman', 'Kopi Hitam', 15_000)))
item_b = CheeseDecorator(MenuFactory.create('makanan', 'Nasi Goreng Spesial', 25_000))

order_1.add_item(item_a, quantity=2)
order_1.add_item(item_b, quantity=1)

print(order_1)


  Order #1 - Budi Santoso
  Waktu  : 20:00:11
  Status : MENUNGGU
  ------------------------------------------------
  2x Kopi Hitam + Susu + Gula            Rp  38,000
  1x Nasi Goreng Spesial + Keju          Rp  30,000
  ------------------------------------------------
  TOTAL                                    Rp  68,000


In [15]:
events = [
    ('Konfirmasi order',  order_1.confirm),
    ('Dapur mulai masak', order_1.start_cooking),
    ('Makanan siap',      order_1.mark_ready),
    ('Transaksi selesai', order_1.complete),
]

for label, action in events:
    print(f'\n  [{label}]')
    action()



  [Konfirmasi order]
    [DAPUR]   Order #1 (Budi Santoso) masuk - mulai memasak!
         -> 2x Kopi Hitam + Susu + Gula
         -> 1x Nasi Goreng Spesial + Keju
    [KASIR]   Order #1 dikonfirmasi - total Rp68,000 siap ditagih.
    [DISPLAY] Nomor antrian #1 - [KUNING] DIKONFIRMASI

  [Dapur mulai masak]
    [DAPUR]   Order #1 (Budi Santoso) masuk - mulai memasak!
         -> 2x Kopi Hitam + Susu + Gula
         -> 1x Nasi Goreng Spesial + Keju
    [DISPLAY] Nomor antrian #1 - [BIRU] DIMASAK

  [Makanan siap]
    [DISPLAY] Nomor antrian #1 - [HIJAU] SIAP

  [Transaksi selesai]
    [KASIR]   Order #1 selesai - transaksi ditutup.
    [DISPLAY] Nomor antrian #1 - [PUTIH] SELESAI


### 3.5 Extensibility — Tambah Observer baru tanpa ubah `OrderSubject`

In [16]:
class WhatsAppObserver(Observer):
    """
    Observer baru: kirim notifikasi WhatsApp ke pelanggan.
    Ditambahkan TANPA mengubah satu baris pun di OrderSubject.
    Inilah Open/Closed Principle pada Observer Pattern.
    """
    def update(self, order: OrderSubject) -> None:
        if order.status == OrderSubject.STATUS_READY:
            print(f'    [WHATSAPP] Halo {order.customer_name}! '
                  f'Order #{order.order_id} Anda sudah siap diambil!')

order_2 = OrderSubject('Siti Rahayu')
order_2.attach(KitchenObserver())
order_2.attach(DisplayObserver())
order_2.attach(WhatsAppObserver())   # <- observer baru, OrderSubject tidak diubah

order_2.add_item(ExtraShot(MenuFactory.create('minuman', 'Es Teh Manis', 8_000)))

print(f'  Order baru: {order_2.customer_name}\n')
order_2.confirm()
order_2.start_cooking()
order_2.mark_ready()


  Order baru: Siti Rahayu

    [DAPUR]   Order #2 (Siti Rahayu) masuk - mulai memasak!
         -> 1x Es Teh Manis + Extra Shot
    [DISPLAY] Nomor antrian #2 - [KUNING] DIKONFIRMASI
    [DAPUR]   Order #2 (Siti Rahayu) masuk - mulai memasak!
         -> 1x Es Teh Manis + Extra Shot
    [DISPLAY] Nomor antrian #2 - [BIRU] DIMASAK
    [DISPLAY] Nomor antrian #2 - [HIJAU] SIAP
    [WHATSAPP] Halo Siti Rahayu! Order #2 Anda sudah siap diambil!


---
<a id='4'></a>
## Demo Gabungan — Ketiga Pattern Bekerja Bersama

Menunjukkan alur lengkap dari satu order nyata:
**Factory** -> buat item -> **Decorator** -> tambah topping -> **Observer** -> sebar notifikasi.


In [17]:
print('=' * 60)
print('  DEMO GABUNGAN - Sistem KantinKu')
print('=' * 60)

# 1. Factory: buat item dasar
print('\n[1] Factory Method: membuat item menu')
burger    = MenuFactory.create('makanan', 'Burger Crispy',    30_000)
milkshake = MenuFactory.create('minuman', 'Milkshake Coklat', 20_000)
print(f'    OK {burger}')
print(f'    OK {milkshake}')

# 2. Decorator: tambah topping
print('\n[2] Decorator: menambahkan topping')
burger_keju      = CheeseDecorator(burger)
shake_susu_extra = ExtraShot(MilkDecorator(milkshake))
print(f'    OK {burger_keju.get_name():<40} Rp{burger_keju.get_price():,}')
print(f'    OK {shake_susu_extra.get_name():<40} Rp{shake_susu_extra.get_price():,}')

# 3. Observer: buat order dan notifikasi
print('\n[3] Observer: proses order dan broadcast notifikasi')
order = OrderSubject('Andi Wijaya')
order.attach(KitchenObserver())
order.attach(CashierObserver())
order.attach(DisplayObserver())
order.attach(WhatsAppObserver())

order.add_item(burger_keju,      quantity=1)
order.add_item(shake_susu_extra, quantity=2)

print()
print(order)

print('\n  --- Lifecycle order ---')
for label, action in [
    ('Konfirmasi', order.confirm),
    ('Dimasak',    order.start_cooking),
    ('Siap',       order.mark_ready),
    ('Selesai',    order.complete),
]:
    print(f'\n  [{label}]')
    action()

print('\n' + '=' * 60)
print('  Demo selesai. Semua pattern berjalan normal.')
print('=' * 60)


  DEMO GABUNGAN - Sistem KantinKu

[1] Factory Method: membuat item menu
    OK [Makanan] Burger Crispy - Rp30,000
    OK [Minuman] Milkshake Coklat - Rp20,000

[2] Decorator: menambahkan topping
    OK Burger Crispy + Keju                     Rp35,000
    OK Milkshake Coklat + Susu + Extra Shot     Rp30,000

[3] Observer: proses order dan broadcast notifikasi

  Order #3 - Andi Wijaya
  Waktu  : 20:00:11
  Status : MENUNGGU
  ------------------------------------------------
  1x Burger Crispy + Keju                Rp  35,000
  2x Milkshake Coklat + Susu + Extra Shot Rp  60,000
  ------------------------------------------------
  TOTAL                                    Rp  95,000

  --- Lifecycle order ---

  [Konfirmasi]
    [DAPUR]   Order #3 (Andi Wijaya) masuk - mulai memasak!
         -> 1x Burger Crispy + Keju
         -> 2x Milkshake Coklat + Susu + Extra Shot
    [KASIR]   Order #3 dikonfirmasi - total Rp95,000 siap ditagih.
    [DISPLAY] Nomor antrian #3 - [KUNING] DIKONFIRMA

---
<a id='5'></a>
## Analisis & Kesimpulan

### Mengapa ketiga pattern ini dipilih?

| Pattern | Masalah yang Diselesaikan | Prinsip OOP |
|---|---|---|
| Factory Method | Client tidak perlu tahu class konkret yang dibuat | Open/Closed Principle |
| Decorator | Menambah perilaku tanpa subclass explosion | Single Responsibility |
| Observer | Loose coupling antara Subject dan penerima notifikasi | Dependency Inversion |

---

### Interaksi antar pattern dalam sistem KantinKu

```
Pelanggan pilih menu
    |
    v
MenuFactory.create()          <- FACTORY METHOD
    |  membuat FoodItem / DrinkItem
    v
MilkDecorator(item)           <- DECORATOR
SugarDecorator(item)
    |  mengakumulasi nama + harga
    v
order.add_item(item)
order.confirm()               <- OBSERVER
    |  _notify() -> KitchenObserver
    |           -> CashierObserver
    +--         -> DisplayObserver
```

---

### Kekuatan desain ini

**Factory Method** — `MenuFactory` tidak pernah di-hardcode ke `FoodItem` atau `DrinkItem`.
Tambah `DessertItem` baru cukup satu baris `MenuFactory.register('dessert', DessertItem)`.

**Decorator** — Dengan 4 topping tersedia, ada 2^4 = 16 kombinasi.
Tanpa Decorator butuh 16 subclass. Dengan Decorator cukup 4 class yang ditumpuk bebas.

**Observer** — `OrderSubject` tidak pernah mengimport `KitchenObserver`.
`WhatsAppObserver` ditambahkan tanpa satu baris pun berubah di `OrderSubject`.
Inilah arti *loose coupling* yang sesungguhnya.
